# Silicon-strip detector analysis

Explore detector calibration, channel noise, source response and cluster-signal distributions using the aggregate tables in `data/derived/`.


In [1]:
from pathlib import Path
import csv, json, sys
root=Path.cwd()
if root.name=='notebooks': root=root.parent
sys.path.insert(0,str(root/'scripts'))
from validate_project import validate

def table(name):
    with (root/'data/derived'/name).open() as handle:
        return list(csv.DictReader(handle))
print('Silicon-strip analysis')


Silicon-strip analysis


## Data provenance

Source IDs link the summaries to export identities, checksums and selection records in `metadata/source_manifest.json`. The result tables have their own schemas and checksums in `metadata/derived_manifest.json`.


In [2]:
manifest=json.loads((root/'metadata/source_manifest.json').read_text())
print('Source records:',len(manifest['inputs']))
specs=json.loads((root/'metadata/derived_manifest.json').read_text())
print('Summary tables:',len(specs))


Source records: 36
Summary tables: 6


## Calibration response

The mean response combines five channel curves on a common charge grid. The zero-charge point remains in the summary. Equivalent energy converts injected charge, rather than applying a validated inverse ADC calibration.

![Calibration](../reports/figures/calibration_response.svg)


In [3]:
calibration=table('calibration_mean_response.csv')
print('Charge settings:',len(calibration))
print('Channels averaged:',sorted({int(row['channel_count']) for row in calibration}))
print('Zero-charge settings:',sum(float(row['injected_charge_electrons'])==0 for row in calibration))


Charge settings: 255
Channels averaged: [5]
Zero-charge settings: 1


## Pedestal and noise

Channel statistics summarize 100 events per channel. The noise figure compares the historical export with noise recomputed after common-mode correction. These quantities have different definitions and sample coverage.

![Pedestal](../reports/figures/pedestal_response.svg)

![Noise](../reports/figures/noise_comparison.svg)


In [4]:
pedestal=table('pedestal_channel_statistics.csv')
print('Channels:',len(pedestal))
print('Events per channel:',sorted({int(row['event_count']) for row in pedestal}))
for column in ['pedestal_adc','common_mode_corrected_sample_noise_adc']:
    values=[float(row[column]) for row in pedestal]
    print(column,'range:',round(min(values),4),'to',round(max(values),4),'ADC')


Channels: 128
Events per channel: [100]
pedestal_adc range: 508.83 to 519.58 ADC
common_mode_corrected_sample_noise_adc range: 1.3831 to 2.2213 ADC


## Source response

Each cluster sums every contributing channel value before clusters are averaged. The historical first-channel reduction is retained for comparison. Voltage and signal remain paired in processing.

![Source response](../reports/figures/source_voltage_response.svg)


In [5]:
for row in table('source_voltage_response.csv')[::5]:
    print(row['bias_voltage_v'],'V:',row['cluster_count'],'clusters;',
          'first channel',round(float(row['historical_mean_first_channel_adc']),3),
          'ADC; cluster sum',round(float(row['mean_cluster_sum_adc']),3),'ADC')


0 V: 200 clusters; first channel 11.719 ADC; cluster sum 12.119 ADC
50 V: 200 clusters; first channel 51.73 ADC; cluster sum 67.288 ADC
100 V: 200 clusters; first channel 67.918 ADC; cluster sum 95.394 ADC
150 V: 200 clusters; first channel 64.087 ADC; cluster sum 96.738 ADC
200 V: 200 clusters; first channel 67.124 ADC; cluster sum 97.021 ADC


## Cluster spectrum

The histogram and summary describe 2,000 complete cluster rows. These are ADC-level results; converting the spectrum to particle energy requires further calibration work.

![ADC spectrum](../reports/figures/large_source_adc_spectrum.svg)


In [6]:
summary=table('large_source_signal_summary.csv')[0]
print('Clusters:',summary['cluster_count'])
print('Mean cluster sum:',round(float(summary['mean_cluster_sum_adc']),4),'ADC')
print('Spectrum bins:',len(table('large_source_adc_spectrum.csv')))


Clusters: 2000
Mean cluster sum: 92.8385 ADC
Spectrum bins: 150


## Laser and current–voltage scans

The laser scan compares responses across neighbouring channels. The current–voltage series shows leakage current versus detector bias.

![Laser position](../reports/figures/laser_position_response.svg)

![Current-voltage](../reports/figures/current_voltage_response.svg)

## Validation

Check the table schemas, numeric values, source references and checksums with `validate()`.


In [7]:
errors,warnings=validate(root)
assert errors==[],errors
print('Schema, numeric and checksum validation: PASS')


Schema, numeric and checksum validation: PASS
